[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/personal-health-agent/binf4070-2026/blob/main/week-02/lab-02.ipynb)

# Week 2 Lab: Make a Readable Visit Summary

**Course:** BINF 4070 — The Future of Personal Health Assistant<br>
**Week:** 2 · Clinical records and FHIR

Make a small card that helps someone review a laboratory result and a medication
order before a visit. We will use one fictional person's records throughout.

By the end, you will be able to:

- find a result and its patient reference in a FHIR record;
- diagnose a summary that uses the wrong source fields;
- decide what to display when status, values, units, or permissions are incomplete;
- select the relevant records and decide whether a numerical comparison is supported.

In Colab, choose **File → Save a copy in Drive**, then run cells from top to
bottom. Each part starts with an example or reminder. Replace the placeholders,
remove any `raise NotImplementedError(...)`
line, and run the check below. Save your outputs as well as your code.

Submit your in-class progress and visible outputs for attendance, even if some
TODOs are incomplete. Complete the labeled take-home in the same notebook.

Everything here is synthetic. You need no EHR account, API key, data download,
or device. Saving a Colab copy in Drive uses your Google account; local Jupyter
needs no sign-in.


In [ ]:
# Run this cell first. Python's standard library is enough; no pip install needed.
import json
from copy import deepcopy
from datetime import datetime


def check_answer(answer, name):
    '''Catch an empty answer or a TODO placeholder, not judge its meaning.'''
    assert isinstance(answer, str) and answer.strip(), f"Complete {name}."
    assert "TODO" not in answer, f"Replace the placeholder in {name}."


def is_number(value):
    '''Recognize the integer and decimal values used in our JSON examples.'''
    return type(value) in (int, float)


print("Ready: all records and exercises run inside this notebook.")


---

## Part 1: Read one person's records

**Reminder from the slides:** FHIR means **Fast Healthcare Interoperability
Resources**. A *resource* is one structured piece of health information. An
Electronic Health Record (**EHR**) can contain many linked resources.

Our packet has three:

| Resource | What it tells us |
|---|---|
| `Patient` | Who the record is about |
| `Observation` | A laboratory result and its context |
| `MedicationRequest` | A medication order; this does not establish that the person took it |

FHIR can represent resources as **JSON**, a text format for named fields and
values. Here those records are already loaded as Python dictionaries. Read a
field with `record["field"]`. Square brackets also surround a **list**; `[0]`
selects its first item. Run the next cell and inspect the printed packet.

This is a Python dictionary holding three resources, not a FHIR Bundle. A Bundle
is FHIR's own collection resource, which we saw in the lecture.


In [ ]:
patient = {
    "resourceType": "Patient",
    "id": "patient-001",
    "name": [{"use": "official", "family": "Rivera", "given": ["Alex"]}],
}

observation = {
    "resourceType": "Observation",
    "id": "a1c-001",
    "status": "final",
    "code": {
        "coding": [{
            "system": "http://loinc.org",
            "code": "4548-4",
            "display": "Hemoglobin A1c/Hemoglobin.total in Blood",
        }],
        "text": "Hemoglobin A1c (HbA1c)",
    },
    "subject": {"reference": "Patient/patient-001"},
    "effectiveDateTime": "2026-09-16T09:00:00-04:00",
    "issued": "2026-09-16T12:00:00-04:00",
    "valueQuantity": {
        "value": 6.2,
        "unit": "%",
        "system": "http://unitsofmeasure.org",
        "code": "%",
    },
}

medication_order = {
    "resourceType": "MedicationRequest",
    "id": "med-order-001",
    "status": "active",
    "intent": "order",
    "medicationCodeableConcept": {
        "coding": [
            {
                "system": "https://clinic.example.org/medications",
                "code": "MET500-TAB",
                "display": "Metformin 500 mg tablet (local catalog)",
            },
            {
                "system": "http://www.nlm.nih.gov/research/umls/rxnorm",
                "code": "861007",
                "display": "metformin hydrochloride 500 MG Oral Tablet",
            },
        ],
        "text": "Metformin 500 mg oral tablet",
    },
    "subject": {"reference": "Patient/patient-001"},
    "authoredOn": "2026-09-15",
}

visit_packet = {
    "patient": patient,
    "result": observation,
    "medication_order": medication_order,
}
print(json.dumps(visit_packet, indent=2))


### A worked example: follow a field path

`resourceType` names the kind of record. `id` identifies this resource within
its type on a server. `subject.reference` points to the person it describes.
We have supplied an `id` for every record in this packet; the base FHIR model
does not require every resource to arrive with one.

Read `observation["code"]["coding"][0]["code"]` one step at a time:

1. `["code"]` opens a dictionary describing the test.
2. `["coding"]` opens its list of terminology entries.
3. `[0]` selects the first item, which is another dictionary.
4. `["code"]` reads the test identifier from that item.

Our list has one coding. Real records can have several; the first is not always
the preferred one. **`system` plus `code` identifies the test**; `display` is
its terminology label and `code.text` is the wording supplied with this record.
LOINC identifies the test, not its measured value: `4548-4` and `6.2` serve
different purposes below.

The slides use `effective[x]` to mean that FHIR allows several time data types.
Our concrete field is **`effectiveDateTime`**, the time the result relates to.
It need not be the upload or report-release time. Keep the timezone (`-04:00`).
Run this complete example.


In [ ]:
test_coding = observation["code"]["coding"][0]
print("Test identity:", test_coding["system"], "|", test_coding["code"])
print("Label:", observation["code"]["text"])
print("Value and unit:", observation["valueQuantity"]["value"], observation["valueQuantity"]["unit"])
print("Result time:", observation["effectiveDateTime"])
print("Status:", observation["status"])
print("Source:", observation["resourceType"] + "/" + observation["id"])


In [ ]:
# Worked warm-up: compare resource references, not people's names.
result_id = observation["id"]
result_patient_reference = observation["subject"]["reference"]
same_patient = result_patient_reference == "Patient/" + patient["id"]


In [ ]:
# Check the extracted values without repeating their dictionary paths.
assert result_id == "a1c-001", "Read the Observation id."
assert result_patient_reference == "Patient/patient-001"
assert same_patient is True, "Does the reference point to this Patient resource?"
print(f"{result_id} points to {result_patient_reference}; patient match: {same_patient}")


### TODO: Find the medication's RxNorm identity

The order carries two terminology entries: a fictional clinic catalog and
**RxNorm**, the medication vocabulary from the slides. Their position in the
list can change. Find the RxNorm entry by its `system`, keeping its `code` and
`display` together. Return that dictionary, or `None` if no RxNorm entry exists.
For these examples, there is at most one RxNorm entry.

Use a `for` loop and an `if` condition. A `return` inside the loop ends the
function immediately; consider where the no-match `return None` belongs.
With a dictionary, `.get("coding", [])` supplies an empty list when `coding`
is absent, so there is simply nothing to loop over. `.get("system")` returns
`None` if that field is absent.
Keep the original source wording separately as `medication_text`.

Finally, name the order's id and explain why an **active order** does not tell
us whether Alex is taking the medication. No medical knowledge is needed:
consider which event this resource records.


In [ ]:
RXNORM_SYSTEM = "http://www.nlm.nih.gov/research/umls/rxnorm"


def find_rxnorm_coding(order):
    '''Return the RxNorm coding dictionary, or None when it is absent.'''
    concept = order["medicationCodeableConcept"]
    # TODO: Inspect concept's coding list and select by system, not position.
    # Hint: Each list item is a dictionary; .get() also works on these items.
    raise NotImplementedError("Complete the lookup and remove this line.")


# TODO: Read the source text from medicationCodeableConcept.text.
medication_text = None
medication_identity = find_rxnorm_coding(medication_order)
# TODO: Explain one limit of this record, naming its id.
medication_limit = "TODO: Write one sentence about the order and actual use."


In [ ]:
expected_identity = {
    "system": "http://www.nlm.nih.gov/research/umls/rxnorm",
    "code": "861007",
    "display": "metformin hydrochloride 500 MG Oral Tablet",
}
assert medication_text == "Metformin 500 mg oral tablet"
assert medication_identity == expected_identity, "Return the whole RxNorm coding."

# The answer must stay the same when the list order changes.
reordered_order = deepcopy(medication_order)
reordered_order["medicationCodeableConcept"]["coding"].reverse()
assert find_rxnorm_coding(reordered_order) == expected_identity
uncoded_order = {"medicationCodeableConcept": {"text": "Source text only"}}
assert find_rxnorm_coding(uncoded_order) is None, "No RxNorm entry means no match."
local_only_order = {"medicationCodeableConcept": {"coding": [{
    "system": "https://clinic.example.org/medications", "code": "861007",
}]}}
assert find_rxnorm_coding(local_only_order) is None, "The digits alone do not identify the system."
check_answer(medication_limit, "medication_limit")
assert "med-order-001" in medication_limit, "Name the record you inspected."
print("Source text:", medication_text)
print("RxNorm coding:", medication_identity)
print("Limit:", medication_limit)
print("Source checks passed. Review the sentence with a partner; code cannot judge its meaning.")


---

## Part 2: Debug a plausible summary

A result such as `6.2` is hard to read alone. We will put its label, unit, time,
status, and source beside it, then add the medication order. Here is the card
our code will produce:

```text
VISIT SUMMARY | Synthetic classroom records
Patient: Alex Rivera (Patient/patient-001)
Hemoglobin A1c (HbA1c): 6.2 %
Result time: 2026-09-16T09:00:00-04:00
Recorded status: final
Result source: Observation/a1c-001
Medication ordered: Metformin 500 mg oral tablet
Order status: active | Written: 2026-09-15
Order source: MedicationRequest/med-order-001
An order does not establish that the medication was taken.
```

### A draft that runs but misleads

A teammate wrote the helper below. It runs without errors, but three choices
produce incorrect card fields. Inspect its output on the original result, a
result recorded in a different unit, and a result with no value.

For this card, `time` means the time the observation relates to:
`effectiveDateTime`. The separate field `issued` says when this version was
made available. These can differ. A missing quantity must remain missing;
the helper must not supply a number or borrow a unit from another record.

Python's `.get("field")` returns `None` if a field is absent. Giving a second
argument changes the fallback: `.get("field", {})` returns an empty dictionary.
Read the defaults in the draft as carefully as its field names.

This helper is for our supplied, single-value Observation examples. It is not
a general FHIR parser: other results can use text, codes, periods, or multiple
components.


In [ ]:
def draft_result_fields(obs):
    '''A teammate's draft: runs, but contains three field-selection mistakes.'''
    quantity = obs.get("valueQuantity", {})
    return {
        "label": obs["code"]["text"],
        "value": quantity.get("value", 0),
        "unit": observation["valueQuantity"]["unit"],
        "time": obs["issued"],
        "status": obs["status"],
        "source_id": obs["id"],
    }


changed_result = deepcopy(observation)
changed_result.update(
    id="glucose-debug", effectiveDateTime="2026-09-17T09:00:00-04:00",
    issued="2026-09-18T12:00:00-04:00",
)
changed_result["code"] = {
    "coding": [{"system": "http://loinc.org", "code": "2345-7",
                "display": "Glucose [Mass/volume] in Serum or Plasma"}],
    "text": "Glucose",
}
changed_result["valueQuantity"] = {
    "value": 94, "unit": "mg/dL", "system": "http://unitsofmeasure.org", "code": "mg/dL",
}
absent_result = deepcopy(observation)
absent_result["id"] = "a1c-no-value"
del absent_result["valueQuantity"]
absent_result["dataAbsentReason"] = {"coding": [{
    "system": "http://terminology.hl7.org/CodeSystem/data-absent-reason", "code": "unknown",
}]}
debug_records = {"original": observation, "changed": changed_result, "absent": absent_result}
draft_outputs = {}
for name, record in debug_records.items():
    draft_outputs[name] = draft_result_fields(record)
    print("\nSource:", json.dumps(record, indent=2))
    print("Draft output:", draft_outputs[name])


### TODO: Diagnose, then repair

Choose one discrepancy you can point to in the printed evidence. In two
sentences, name the record, quote its incorrect output, identify the source
value or missing field that contradicts it, and explain what caused the mistake.
“The output is wrong” is not a diagnosis: identify the wrong source or default.

Then repair **all three mistakes** in `result_fields` below. Keep its six output
keys and use the input `obs`. Leave `draft_result_fields` unchanged so your
notebook preserves the before-and-after evidence. We are preserving recorded
values and units, not converting between them.


In [ ]:
# TODO: Diagnose one failure using the draft's printed output and its source record.
debug_diagnosis = "TODO: Name the record, wrong output, correct source value, and cause."


In [ ]:
# TODO: Repair the helper; the three mistakes are still present in this copy.
def result_fields(obs):
    '''Extract six card fields from one supplied synthetic Observation.'''
    quantity = obs.get("valueQuantity", {})
    return {
        "label": obs["code"]["text"],
        "value": quantity.get("value", 0),
        "unit": observation["valueQuantity"]["unit"],
        "time": obs["issued"],
        "status": obs["status"],
        "source_id": obs["id"],
    }


fields = result_fields(observation)
print(fields)


In [ ]:
check_answer(debug_diagnosis, "debug_diagnosis")
assert fields == {
    "label": "Hemoglobin A1c (HbA1c)", "value": 6.2, "unit": "%",
    "time": "2026-09-16T09:00:00-04:00", "status": "final", "source_id": "a1c-001",
}
assert result_fields(changed_result) == {
    "label": "Glucose", "value": 94, "unit": "mg/dL",
    "time": "2026-09-17T09:00:00-04:00", "status": "final", "source_id": "glucose-debug",
}
absent_fields = result_fields(absent_result)
assert absent_fields["value"] is None and absent_fields["unit"] is None
for name, record in debug_records.items():
    print(name, "| before:", draft_outputs[name], "| after:", result_fields(record))
print("Repairs pass these cases. Review the diagnosis against the printed before-and-after evidence.")


### Run the provided card display

Run this function without editing it. It checks that both records point to Alex,
then prints the card. The first version displays our complete result. In Part 3,
we will supply a formatting function that also handles other result states.

The source id helps us find the original record; it does not prove that the
record is correct. This notebook keeps the original resources in `visit_packet`.


In [ ]:
def show_visit_summary(person, obs, order, result_formatter=None, note=""):
    '''Print a card for our supplied records; optional formatter changes one line.'''
    patient_reference = "Patient/" + person["id"]
    assert obs["subject"]["reference"] == patient_reference, "Result belongs to another patient."
    assert order["subject"]["reference"] == patient_reference, "Order belongs to another patient."
    result = result_fields(obs)
    name = " ".join(person["name"][0]["given"]) + " " + person["name"][0]["family"]
    if result_formatter is None:
        result_line = f"{result['value']} {result['unit']}"
    else:
        result_line = result_formatter(result)
    lines = [
        "VISIT SUMMARY | Synthetic classroom records",
        f"Patient: {name} ({patient_reference})",
        f"{result['label']}: {result_line}",
        f"Result time: {result['time']}",
        f"Recorded status: {result['status']}",
        f"Result source: Observation/{result['source_id']}",
        f"Medication ordered: {order['medicationCodeableConcept']['text']}",
        f"Order status: {order['status']} | Written: {order['authoredOn']}",
        f"Order source: MedicationRequest/{order['id']}",
        "An order does not establish that the medication was taken.",
    ]
    if note:
        lines.append("Reader note: " + note)
    card = "\n".join(lines)
    print(card)
    return card


guided_card = show_visit_summary(patient, observation, medication_order)


---

## Part 3: Keep incomplete results clear

**Reminder from the slides:** a record's status and missing fields affect what
we can say. A number alone is not enough.

- **Final with a value:** the result is complete. Final does not
  mean normal, and a later correction is still possible.
- **Preliminary with a value:** an initial or interim result. Keep that label
  visible even when its number matches a final result.
- **Observation present, value missing:** we received the record, but it supplies
  no number. That differs from receiving no Observation at all. Our example uses
  `dataAbsentReason` with code `unknown`: an expected value is unknown. It does
  not establish why the value is missing.

As in Week 1, **missing is not zero**. A final status and an absent value can
coexist: completion status and value availability answer different questions.

The next cell makes copies of the same synthetic Observation. These are
alternative examples, not three measurements to compare as a trend.


In [ ]:
final_result = deepcopy(observation)
preliminary_result = deepcopy(observation)
preliminary_result["id"] = "a1c-preliminary"
preliminary_result["status"] = "preliminary"

missing_result = deepcopy(observation)
missing_result["id"] = "a1c-missing"
del missing_result["valueQuantity"]
missing_result["dataAbsentReason"] = {
    "coding": [{
        "system": "http://terminology.hl7.org/CodeSystem/data-absent-reason",
        "code": "unknown",
        "display": "Unknown",
    }],
}

for case in [final_result, preliminary_result, missing_result]:
    print(case["id"], "|", case["status"], "|", case.get("valueQuantity", "No valueQuantity"))
print("Missing-value explanation:", missing_result["dataAbsentReason"])


### TODO: Decide which message takes priority

Write the conditions as well as the returned strings. Apply this classroom
display policy to the fields returned by your repaired helper:

| Situation | Required display |
|---|---|
| Status is neither `final` nor `preliminary` | Say **not displayed** and name the status; show no value. This rule takes priority, even if the value is missing. |
| Supported status, but value is `None` | Say **Result unavailable** and retain the status. Do not invent a number. |
| Supported status and a value, but unit is absent or empty | Show the recorded value, **unit unavailable**, and status. Do not guess a unit. |
| Supported status, value, and unit | Show value, unit, and status. A preliminary value must stay labeled preliminary. |

Two conditions may hold at once. Work out which message should win before
writing the branches. Missing is represented by `None`; zero is a value.
The artificial-zero check tests programming behavior, not clinical meaning.
You may use an f-string such as `f"{result['value']} {unit_text} ({status})"`.


In [ ]:
def safe_result_line(result):
    '''Apply the display policy to one six-field result dictionary.'''
    # TODO: Write branches that satisfy the policy, including overlapping cases.
    raise NotImplementedError("Implement the display policy and remove this line.")


In [ ]:
final_line = safe_result_line(result_fields(final_result))
preliminary_line = safe_result_line(result_fields(preliminary_result))
missing_line = safe_result_line(result_fields(missing_result))
for line in [final_line, preliminary_line, missing_line]:
    check_answer(line, "safe_result_line")
assert "final" in final_line.lower() and "preliminary" in preliminary_line.lower()
for line in [final_line, preliminary_line]:
    assert "6.2" in line and "%" in line
assert "unavailable" in missing_line.lower(), "Say that the result is unavailable."
assert not any(character.isdigit() for character in missing_line), "Do not invent a missing value."

# Programming edge case only; this is not an additional clinical measurement.
zero_fields = dict(result_fields(final_result), value=0)
zero_line = safe_result_line(zero_fields)
assert "0" in zero_line and "unavailable" not in zero_line.lower(), "Zero is not missing."

# Cases where two rules could apply at once.
preliminary_missing = dict(result_fields(missing_result), status="preliminary")
preliminary_missing_line = safe_result_line(preliminary_missing)
assert "unavailable" in preliminary_missing_line.lower() and "preliminary" in preliminary_missing_line.lower()
unsupported_missing = dict(result_fields(missing_result), status="entered-in-error")
unsupported_line = safe_result_line(unsupported_missing)
assert "not displayed" in unsupported_line.lower() and "entered-in-error" in unsupported_line.lower()
unsupported_numeric_line = safe_result_line(dict(result_fields(final_result), status="cancelled"))
assert "not displayed" in unsupported_numeric_line.lower() and "6.2" not in unsupported_numeric_line
for absent_unit in [None, ""]:
    unit_line = safe_result_line(dict(result_fields(preliminary_result), unit=absent_unit))
    assert "6.2" in unit_line and "unit unavailable" in unit_line.lower() and "preliminary" in unit_line.lower()

print("Final:", final_line)
print("Preliminary:", preliminary_line)
print("Missing:", missing_line)
print("Artificial zero check:", zero_line)
print("Preliminary and missing:", preliminary_missing_line)
print("Unsupported and missing:", unsupported_line)
print("Value with no unit:", unit_line)
print("\nCard with missing result:")
missing_card = show_visit_summary(patient, missing_result, medication_order, safe_result_line)


### TODO: Respond to a narrower permission grant

**Reminder from the slides:** SMART on FHIR uses *scopes* to describe requested
access. Read `patient/Observation.rs` as: for the **patient** in context, access
**Observation** records by **reading** (`r`) and **searching** (`s`). Reading
fetches one known record; searching asks for matching records.

The app requested `patient/Observation.rs`, but the authorization server
granted only `patient/Observation.r`. Decide whether each proposed action is
permitted **by that grant**:

- Read the known record `Observation/a1c-001` for the patient in context.
- Search for that patient's latest glucose results.
- Update an Observation to correct a value (`u` means update).

The user clicks **Find recent results**. Write the message the app should show:
explain the permission limit and a useful next step without claiming that the
patient has no results. This exercise concerns Observation access; other resources need
their own permissions. No server call or scope parser is needed.


In [ ]:
requested_scope = "patient/Observation.rs"
granted_scope = "patient/Observation.r"
# TODO: Replace each None with True or False, using the granted scope.
granted_actions = {"read_known": None, "search": None, "update": None}
# TODO: Explain why search is disabled and what the user can do next.
disabled_search_message = "TODO: Write a useful message for the app's user."


In [ ]:
assert set(granted_actions) == {"read_known", "search", "update"}
assert all(type(answer) is bool for answer in granted_actions.values()), "Decide all three actions."
check_answer(disabled_search_message, "disabled_search_message")
print("Requested:", requested_scope, "| Granted:", granted_scope)
print("Action decisions:", granted_actions)
print("App message:", disabled_search_message)
print("Completion check only: review the decisions against the grant and the message for usefulness.")
print("These assertions do not determine whether your permission decisions are correct.")


---

## Take-home TODO: Should this card show a difference?

A small app has received an unordered list of results. Before Alex's visit,
it needs the **most recent glucose Observation for Alex**. A teammate also
wants a line showing the difference from the preceding result. Some incoming
records belong to another patient or describe another test. One uses the same
code characters in a different terminology system.

Write a short selection function, then reuse your existing card helpers.
Match the **patient reference** and the **system + code** pair before comparing
result times. Search every coding entry; its position can vary. The newest
matching record may be preliminary or lack a value: keep that record and let
your formatter explain its state. Substituting an older numeric result would
hide the latest result for that test.

Next, use a provided wrapper to find the immediately preceding matching
record. Decide whether the two records meet a stated comparison policy.
Finally, invent one small case where subtraction works but should not appear
on the card. The work builds toward a card with traceable records and an
honest explanation whenever a comparison is withheld.

Here, “most recent” means the greatest `effectiveDateTime`, not the last item
in the list. All supplied times include a timezone, and matching times are
unique. We are practicing this one selection rule, not resolving corrected
reports or implementing every possible FHIR time field.

Submit the complete notebook, including the take-home output and the final
reflection, **by the following Wednesday**.


In [ ]:
# Synthetic records from an incoming-results queue, deliberately out of order.
incoming_results = [
    {
        "resourceType": "Observation", "id": "glucose-new", "status": "preliminary",
        "code": {
            "coding": [
                {"system": "https://clinic.example.org/tests", "code": "GLU"},
                {"system": "http://loinc.org", "code": "2345-7",
                 "display": "Glucose [Mass/volume] in Serum or Plasma"},
            ],
            "text": "Glucose",
        },
        "subject": {"reference": "Patient/patient-001"},
        "effectiveDateTime": "2026-09-18T09:00:00-04:00",
        "valueQuantity": {"value": 97, "unit": "mg/dL",
                          "system": "http://unitsofmeasure.org", "code": "mg/dL"},
    },
    # Same patient, different test.
    deepcopy(observation),
    {
        "resourceType": "Observation", "id": "glucose-local", "status": "final",
        "code": {
            "coding": [{"system": "https://clinic.example.org/tests", "code": "2345-7"}],
            "text": "Local glucose entry",
        },
        "subject": {"reference": "Patient/patient-001"},
        "effectiveDateTime": "2026-09-21T09:00:00-04:00",
        "valueQuantity": {"value": 99, "unit": "mg/dL",
                          "system": "http://unitsofmeasure.org", "code": "mg/dL"},
    },
    {
        "resourceType": "Observation", "id": "glucose-other-patient", "status": "final",
        "code": {
            "coding": [{"system": "http://loinc.org", "code": "2345-7",
                        "display": "Glucose [Mass/volume] in Serum or Plasma"}],
            "text": "Glucose",
        },
        "subject": {"reference": "Patient/patient-002"},
        "effectiveDateTime": "2026-09-20T09:00:00-04:00",
        "valueQuantity": {"value": 101, "unit": "mg/dL",
                          "system": "http://unitsofmeasure.org", "code": "mg/dL"},
    },
    {
        "resourceType": "Observation", "id": "glucose-old", "status": "final",
        "code": {
            "coding": [{"system": "http://loinc.org", "code": "2345-7",
                        "display": "Glucose [Mass/volume] in Serum or Plasma"}],
            "text": "Glucose",
        },
        "subject": {"reference": "Patient/patient-001"},
        "effectiveDateTime": "2026-09-16T09:00:00-04:00",
        "valueQuantity": {"value": 94, "unit": "mg/dL",
                          "system": "http://unitsofmeasure.org", "code": "mg/dL"},
    },
]
print(json.dumps(incoming_results, indent=2))


def result_time(obs):
    '''Convert our supplied timezone-aware timestamp into a comparable time.'''
    return datetime.fromisoformat(obs["effectiveDateTime"])


# A worked reminder: Python can compare these converted times directly.
print("First record is newer than the last:",
      result_time(incoming_results[0]) > result_time(incoming_results[-1]))


### TODO: Select the latest matching record

Complete the function below. Return the matching Observation dictionary with
the latest result time, or `None` if no record matches. Do not change the input
records. A loop, a few conditions, and one variable holding the best match so
far are enough; no new package is needed.

Use `result_time(record)` when comparing times. First decide whether a record
belongs to the requested patient and has the requested terminology entry.
Remember that both parts of the terminology identity must occur in the
**same** coding entry. Do not filter by status or whether a number is present.


In [ ]:
def select_latest_result(records, patient_reference, system, code):
    '''Return the newest matching Observation, or None when none match.'''
    latest = None
    # TODO: Inspect each record, keep matches, and update latest when appropriate.
    # Hint: A first match replaces None; later matches must have a newer time.
    raise NotImplementedError("Complete the selection and remove this line.")
    return latest


selected_result = select_latest_result(
    incoming_results, "Patient/patient-001", "http://loinc.org", "2345-7"
)


In [ ]:
assert selected_result is not None, "There is a matching result in this queue."
assert selected_result["id"] == "glucose-new", "Check patient, terminology identity, and time."
assert select_latest_result(
    list(reversed(incoming_results)), "Patient/patient-001", "http://loinc.org", "2345-7"
)["id"] == "glucose-new", "List order must not change the answer."
assert select_latest_result(
    incoming_results, "Patient/patient-002", "http://loinc.org", "2345-7"
)["id"] == "glucose-other-patient", "Use the requested patient."
assert select_latest_result(
    incoming_results, "Patient/patient-001", "http://loinc.org", "4548-4"
)["id"] == "a1c-001", "Use the requested test."
assert select_latest_result(
    incoming_results, "Patient/patient-001", "https://clinic.example.org/tests", "2345-7"
)["id"] == "glucose-local", "Use the requested terminology system."
assert select_latest_result(
    incoming_results, "Patient/not-in-queue", "http://loinc.org", "2345-7"
) is None
assert select_latest_result([], "Patient/patient-001", "http://loinc.org", "2345-7") is None

# A missing newest value must not cause the older numeric result to win.
missing_queue = deepcopy(incoming_results)
for record in missing_queue:
    if record["id"] == "glucose-new":
        record.pop("valueQuantity")
        record["dataAbsentReason"] = deepcopy(missing_result["dataAbsentReason"])
selected_missing = select_latest_result(
    missing_queue, "Patient/patient-001", "http://loinc.org", "2345-7"
)
assert selected_missing["id"] == "glucose-new", "Keep the newest match even without a number."
assert "unavailable" in safe_result_line(result_fields(selected_missing)).lower()
print("Selection checks passed: patient, test, system, time, no match, and missing value.")


In [ ]:
def select_result_pair(records, patient_reference, system, code):
    '''Return the latest and immediately preceding matches before judging either.'''
    latest = select_latest_result(records, patient_reference, system, code)
    if latest is None:
        return None, None
    earlier_records = [record for record in records if result_time(record) < result_time(latest)]
    previous = select_latest_result(earlier_records, patient_reference, system, code)
    return latest, previous


selected_result, previous_result = select_result_pair(
    incoming_results, "Patient/patient-001", "http://loinc.org", "2345-7"
)
print("Selected pair:", selected_result["id"], "and", previous_result["id"])


In [ ]:
# Reuse the card and formatter you completed in class.
if selected_result is None:
    takehome_card = "No matching record returned for this patient and test."
    print(takehome_card)
else:
    takehome_card = show_visit_summary(
        patient, selected_result, medication_order, safe_result_line
    )

# A separate query shows how no match differs from a record with no value.
no_match = select_latest_result(
    incoming_results, "Patient/not-in-queue", "http://loinc.org", "2345-7"
)
if no_match is None:
    print("\nNo matching record returned for the second query.")
else:
    print("Unexpected match; inspect the selection logic.")


### TODO: Withhold comparisons that fail the policy

The supplied wrapper returns a pair: `pair[0]` is the latest record and
`pair[1]` is its predecessor. The notation `compare_result_pair(*pair)` passes
those two records as the function's two arguments.

For this classroom card, show **latest minus previous** only when both
selected records are final, both supply numbers, and their recorded units are
the same nonempty string. Otherwise withhold the difference and explain why.
Use the most recent pair already selected. Do not search further back for a
more convenient comparison, convert units, or change the records.

This is a deliberately limited **display policy**, not proof that two clinical
measurements are comparable. A numerical difference alone establishes no
trend, diagnosis, improvement, or deterioration. The latest card stays visible
even when its comparison is withheld.

Return a dictionary with exactly these fields:

| Field | Allowed comparison | Withheld comparison |
|---|---|---|
| `delta` | Latest value minus previous value | `None` |
| `unit` | Their common recorded unit | `None` |
| `reason` | Briefly explain why the policy permits this pair | Name the reason the policy blocks it |

Consider failure reasons in this order: absent record, nonfinal status,
missing or nonnumeric value, then absent or different units. An absent record
is `None`; a missing numeric value is inside an existing record. Use
`result_fields` for extraction and the provided `is_number(value)` to test
numbers. It accepts our integers and decimals, including zero; strings such
as `"97"` are not numbers in this exercise.


In [ ]:
def compare_result_pair(latest, previous):
    '''Return delta, unit, and reason for the two already-selected Observations.'''
    # TODO: Apply the comparison policy; keep selection separate from this decision.
    raise NotImplementedError("Implement the comparison policy and remove this line.")


In [ ]:
# These are alternative software test cases, not additional patient history.
selected_comparison = compare_result_pair(selected_result, previous_result)

# Change only the status of the newest result to make a permitted comparison.
finalized_queue = deepcopy(incoming_results)
for record in finalized_queue:
    if record["id"] == "glucose-new":
        record["status"] = "final"
finalized_pair = select_result_pair(
    finalized_queue, "Patient/patient-001", "http://loinc.org", "2345-7"
)
finalized_comparison = compare_result_pair(*finalized_pair)

# Trap: the immediate predecessor lacks a number; an older usable result exists.
gap_latest = deepcopy(finalized_pair[0])
gap_previous = deepcopy(finalized_pair[1])
gap_previous["id"] = "glucose-gap"
del gap_previous["valueQuantity"]
gap_previous["dataAbsentReason"] = deepcopy(missing_result["dataAbsentReason"])
gap_oldest = deepcopy(finalized_pair[1])
gap_oldest["id"] = "glucose-oldest"
gap_oldest["effectiveDateTime"] = "2026-09-14T09:00:00-04:00"
gap_queue = [gap_oldest, gap_latest, gap_previous]
gap_pair = select_result_pair(gap_queue, "Patient/patient-001", "http://loinc.org", "2345-7")
gap_comparison = compare_result_pair(*gap_pair)


def show_comparison(pair, comparison):
    '''Print the selected sources beside the policy decision.'''
    sources = [record["id"] if record is not None else "no record" for record in pair]
    print("Pair:", " -> ".join(sources))
    if comparison["delta"] is None:
        print("Comparison withheld:", comparison["reason"])
    else:
        print(f"Recorded difference: {comparison['delta']:+g} {comparison['unit']}")
        print("Policy check:", comparison["reason"])


for name, pair, result in [
    ("Original queue", (selected_result, previous_result), selected_comparison),
    ("Finalized test case", finalized_pair, finalized_comparison),
    ("Missing predecessor test case", gap_pair, gap_comparison),
]:
    print("\n" + name)
    show_comparison(pair, result)


In [ ]:
for comparison in [selected_comparison, finalized_comparison, gap_comparison]:
    assert set(comparison) == {"delta", "unit", "reason"}
    check_answer(comparison["reason"], "comparison reason")
assert selected_comparison["delta"] is None and selected_comparison["unit"] is None
assert finalized_comparison["delta"] == 3 and finalized_comparison["unit"] == "mg/dL"
assert gap_pair[1]["id"] == "glucose-gap", "Do not skip the immediately preceding match."
assert gap_comparison["delta"] is None and gap_comparison["unit"] is None
assert compare_result_pair(selected_result, None)["delta"] is None
print("Comparison checks passed. Read each reason: assertions do not judge its explanation.")


### TODO: Make arithmetic fail as a design choice

The function below subtracts two values without inspecting anything else.
Invent **one counterexample** by copying the permitted `finalized_pair` and
changing the **previous record's status** or **either record's units** so the
comparison must be blocked. Keep
both numbers present, keep the same patient and test, and preserve the originals.
The point is to make naive subtraction run successfully while producing a
number the card should not show.

Before running either function, predict the number naive subtraction will
produce and explain why it should be withheld. The original queue already
tests a preliminary newest record; make a different case. You may change the
previous status, remove a unit, or create a different-unit test case.
If changing a unit, keep `valueQuantity.unit` and its coded `code`
consistent; invented values here are software tests, not unit conversions.

The cell after yours runs both functions. Keep its output as evidence, and
revise your comparison helper if it accepts the pair you intended to block.


In [ ]:
counterexample_pair = deepcopy(finalized_pair)
# TODO: Change a relevant field in one copied record; keep both numeric values.

# TODO: Predict the numeric output of naive subtraction before running it.
counterexample_prediction = None
# TODO: Identify your changed field and explain why it should affect display.
counterexample_reason = "TODO"


In [ ]:
def naive_delta(latest, previous):
    '''Deliberately flawed: arithmetic only, without checking record context.'''
    return latest["valueQuantity"]["value"] - previous["valueQuantity"]["value"]


assert is_number(counterexample_prediction), "Record a numeric prediction first."
check_answer(counterexample_reason, "counterexample_reason")
assert counterexample_pair != finalized_pair, "Change a relevant field in the copy."
assert all(is_number(result_fields(record)["value"]) for record in counterexample_pair), "Keep both numbers."
counterexample_output = compare_result_pair(*counterexample_pair)
naive_output = naive_delta(*counterexample_pair)
print("Prediction:", counterexample_prediction, "|", counterexample_reason)
print("Naive subtraction returns:", naive_output)
show_comparison(counterexample_pair, counterexample_output)
counterexample_card = show_visit_summary(patient, counterexample_pair[0], medication_order, safe_result_line)
assert counterexample_prediction == naive_output, "Compare your prediction with the arithmetic."
assert counterexample_output["delta"] is None, "Design a case the policy blocks; then check your helper."
print("Review the changed fields against the reason. Passing arithmetic is not permission to display it.")


### TODO: Explain the display decision

Write a short note to your teammate comparing **one allowed output and one
blocked output** from your notebook. Name the selected record ids, quote the
numeric difference in the allowed case, and identify the field that blocks
the other case. Use your counterexample if it makes the distinction clearer.
Explain why the blocked case should keep its newest result card without a
difference line. Refer to your actual output; do not interpret the numbers
as a health trend.


In [ ]:
# TODO: Use one allowed and one blocked output to explain the card's behavior.
comparison_note = "TODO: Write three or four sentences for your teammate."


In [ ]:
check_answer(comparison_note, "comparison_note")
print("Teammate note:", comparison_note)
print("Completion check only: verify the named ids, number, and blocking field against your outputs.")


## Reference notes

The records in this notebook are invented for this class. Their field names
follow [FHIR R4 Observation](https://hl7.org/fhir/R4/observation.html) and
[MedicationRequest](https://hl7.org/fhir/R4/medicationrequest.html). See the
[Observation status definitions](https://hl7.org/fhir/R4/codesystem-observation-status.html)
and [data-absent-reason definitions](https://hl7.org/fhir/R4/codesystem-data-absent-reason.html)
for the labels used in Part 3. The [Observation time definitions](https://hl7.org/fhir/R4/observation-definitions.html#Observation.issued)
distinguish the time of the observation from when a version was made available.

The test identifiers are LOINC [4548-4](https://loinc.org/4548-4/) and
[2345-7](https://loinc.org/2345-7/). The medication coding is RxNorm
[861007](https://rxnav.nlm.nih.gov/REST/rxcui/861007/properties.json); its normalized
name differs slightly from the source text. The clinic catalog codes are invented.
Permission syntax follows [SMART App Launch 2.2 scopes](https://hl7.org/fhir/smart-app-launch/STU2.2/scopes-and-launch-context.html).
For a Python refresher, see the official guide to
[dictionaries](https://docs.python.org/3/tutorial/datastructures.html#dictionaries).


## Troubleshooting

| What you see | What to try |
|---|---|
| `NameError` | Run setup and earlier cells again, especially after a runtime restart. |
| `NotImplementedError` | Complete the named TODOs, remove that line, and rerun the cell and its check. |
| `AssertionError` | Read the message and compare your field path or output with the original record. |
| A card still shows an old answer | Rerun your edited cell, then rerun the cell that displays the card. |
| You are unsure what a value means medically | Describe the recorded value and its limits; this lab does not require a medical interpretation. |

Before the Wednesday submission, restart the runtime and run your completed
notebook from top to bottom. Save the notebook with the cards and answers visible.


## TODO: Summary and reflection

1. What did you learn about turning a FHIR result into a readable summary without losing its context? Use one record or output from your notebook.
2. How did you use AI in this lab, and what did you check yourself? If you did not use AI, say so and describe one check you made.


In [ ]:
# TODO: Answer question 1 using one of your records or outputs.
week02_learning = "TODO"
# TODO: Answer question 2 about AI use and your own checking.
week02_ai_use = "TODO"

check_answer(week02_learning, "week02_learning")
check_answer(week02_ai_use, "week02_ai_use")
print("What I learned:", week02_learning)
print("AI use and checking:", week02_ai_use)
